# 💼 The Analyst's Notebook · Part 9
### A threshold, and a label that is rare

Part 8 gave the risk report its first classifier. On Apple it ranked the test days well, with an AUC of 0.797, and at a threshold of one half it called a rise on 410 of the 482 test days, which is 85% of them. Buying protection 85% of the time is not a policy anyone would sign off, and Part 8 left the question of where the threshold should sit open.

Part 9 answers it, and changes the label while it is there. The desk does not care about every rise; it cares about a **jump**, a month at least 1.5 times as volatile as the one before. That label is rare, and a rare label breaks accuracy in a way that is worth seeing once on your own data.

## How to work through this

- Run the **quick load** cell first. It brings back what Part 8 established and loads the price table.
- Each question builds on the last, so keep them in order and keep your variables. Later questions use the names earlier ones created.
- Cells with `...` are blanks. The notebook runs cleanly even before you fill them in, so **Run all** is always safe.
- Hints and solutions are folded under each question. Work first, then check.

**A note on units.** The lecture worked in percent and on two different tables. This notebook keeps the plain decimals of Parts 1 to 8 and stays on Apple, so every number here can be compared with Part 8's directly.

*Stuck for more than 15 minutes? Ask a friend, ask an AI for a hint (not the answer), or email me at `jobo@econ.au.dk`.*

---

## ⚙️ Quick load

The packages, the price table, and what Part 8 left you. Run it and read what it prints.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score, recall_score,
                             roc_auc_score, average_precision_score, brier_score_loss,
                             f1_score, classification_report)
from sklearn.model_selection import cross_val_score, TimeSeriesSplit, GridSearchCV

CANDIDATE_DIRS = ["data", os.path.join("..", "data"), "."]
REPO_RAW_URL = "https://raw.githubusercontent.com/theill95/mlfin-2026/main/data/"   # used when the CSV files are not next to the notebook


def data_path(filename):
    """Where the course CSV files are, wherever you happen to be running."""
    for folder in CANDIDATE_DIRS:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    if REPO_RAW_URL is not None:
        return REPO_RAW_URL + filename
    raise FileNotFoundError(
        f"Could not find {filename}. Run this notebook from the course folder, "
        f"upload the CSV into Colab, or set REPO_RAW_URL."
    )


# The whole universe: eleven instruments, 2015 to 2024, returns in plain decimals
prices = pd.read_csv(data_path("prices.csv"), parse_dates=["date"])
wide = prices.pivot(index="date", columns="ticker", values="close")
rets = wide.pct_change()
TICKERS = sorted(prices["ticker"].unique())

folds = TimeSeriesSplit(n_splits=5)

# --- What Part 8 established, on Apple ---
part8_label = "rising: next 20 days more volatile than the last 20"
part8_auc = 0.7971                  # one column, vol_20d, on the test block
part8_accuracy = 0.5705             # at a threshold of one half
part8_majority = 0.5456             # predicting the more common label every day
part8_days_called = 410                # test days it called a rise, of 482
part8_split = "by date: train to 2022-12-31, test from 2023-01-01"

print("Loaded prices:", prices.shape[0], "rows")
print("Instruments  :", ", ".join(TICKERS))
print()
print("Part 8 left you a classifier on Apple:")
print("  label   :", part8_label)
print("  split   :", part8_split)
print(f"  test AUC {part8_auc:.3f}, accuracy {part8_accuracy:.3f} against {part8_majority:.3f} for the majority rule")
print(f"  it called a rise on {part8_days_called} of 482 test days, which is {part8_days_called / 482:.0%} of them")
print()
print("Open question from Part 8: where should the threshold sit?")

---

### Q1 · Where Part 8 stopped

Rebuild Part 8's table for Apple as `table`: the six volatility windows `[5, 10, 20, 40, 60, 120]` as `vol_<w>d`, the three return windows `[5, 20, 60]` as `ret_<w>d`, `up_20d`, the 20-day volatility of every **other** instrument as `<ticker>_vol`, the target `vol_next`, and incomplete rows dropped. Store the twenty feature names in `columns`. Then add Part 8's `rising` label, split at the end of 2022, refit the one-column classifier in a scaled pipeline and check its test AUC matches `part8_auc`.

In [ ]:
table = pd.DataFrame()

for w in [5, 10, 20, 40, 60, 120]:
    ...

for w in [5, 20, 60]:
    ...

table['up_20d'] = ...

for t in TICKERS:
    ...

table['vol_next'] = ...
table = ...
columns = ...

# Part 8's label and split
...
train = ...
test = ...

check = ...
print(check)
print('matches Part 8:', ...)

<details>
<summary>💡 Hint 1</summary>

Column names are text built from the number: `'vol_' + str(w) + 'd'`. Inside the last loop, `if t != 'AAPL':`. `up_20d` is `(rets['AAPL'] > 0).rolling(20).mean()`.

</details>

<details>
<summary>💡 Hint 2</summary>

`columns = list(table.columns[:-1])` before the label is added. The check is `abs(check - part8_auc) < 0.001`.

</details>

<details>
<summary>✅ Solution</summary>

```python
table = pd.DataFrame()

for w in [5, 10, 20, 40, 60, 120]:
    table['vol_' + str(w) + 'd'] = rets['AAPL'].rolling(w).std()

for w in [5, 20, 60]:
    table['ret_' + str(w) + 'd'] = rets['AAPL'].rolling(w).mean()

table['up_20d'] = (rets['AAPL'] > 0).rolling(20).mean()

for t in TICKERS:
    if t != 'AAPL':
        table[t + '_vol'] = rets[t].rolling(20).std()

table['vol_next'] = rets['AAPL'].rolling(20).std().shift(-20)
table = table.dropna()
columns = list(table.columns[:-1])

table['rising'] = (table['vol_next'] > table['vol_20d']).astype(int)
train = table.loc[:'2022-12-31']
test = table.loc['2023-01-01':]

model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(train[['vol_20d']], train['rising'])
check = round(roc_auc_score(test['rising'], model.predict_proba(test[['vol_20d']])[:, 1]), 4)
print(check)
print('matches Part 8:', abs(check - part8_auc) < 0.001)
```

0.7971, and `True`: 2,376 rows, 1,894 to fit on and 482 to check on, exactly as in Part 8. The table and the split carry forward unchanged; only the label is about to move.

</details>

---

### Q2 · A label the desk would act on

A rise of one percent and a rise of 60 percent both counted as `rising`. Add a rarer label, `jump`: 1 where `vol_next` is more than **1.5 times** `vol_20d`. Split again so `train` and `test` carry it, and print the share of jumps in each half and the accuracy of predicting 0 every day, stored as `majority`.

In [ ]:
# add the jump label to table, then split again
...
train = ...
test = ...

print('train:', ...)
print('test :', ...)
majority = ...
print('majority rule:', majority)

<details>
<summary>💡 Hint 1</summary>

`(table['vol_next'] > 1.5 * table['vol_20d']).astype(int)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`majority = 1 - test['jump'].mean()`, because 0 is now much the more common label.

</details>

<details>
<summary>✅ Solution</summary>

```python
table['jump'] = (table['vol_next'] > 1.5 * table['vol_20d']).astype(int)
train = table.loc[:'2022-12-31']
test = table.loc['2023-01-01':]

print('train:', round(train['jump'].mean(), 4))
print('test :', round(test['jump'].mean(), 4))
majority = round(1 - test['jump'].mean(), 4)
print('majority rule:', majority)
```

0.1848 of the training days and 0.0871 of the test days, which is 42 jumps in the two test years against 350 in the eight training years. The rule to beat has moved from 0.546 to 0.9129. Note also that the label is rarer in the test years than in the training years, which matters later.

</details>

---

### Q3 · The same model, the new label

Fit the one-column scaled pipeline on `vol_20d` predicting `jump`, store the test probabilities as `p_jump`, and print the accuracy beside `majority` from Q2. Store the accuracy as `acc_jump`.

In [ ]:
jump_model = ...
...
p_jump = ...
acc_jump = ...

print('accuracy:', acc_jump)
print('majority:', majority)

<details>
<summary>💡 Hint 1</summary>

The pipeline is the one from Q1 with `train['jump']` as the target.

</details>

<details>
<summary>💡 Hint 2</summary>

`acc_jump = round(accuracy_score(test['jump'], jump_model.predict(test[['vol_20d']])), 4)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
jump_model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
jump_model.fit(train[['vol_20d']], train['jump'])
p_jump = jump_model.predict_proba(test[['vol_20d']])[:, 1]
acc_jump = round(accuracy_score(test['jump'], jump_model.predict(test[['vol_20d']])), 4)

print('accuracy:', acc_jump)
print('majority:', majority)
```

0.9129 and 0.9129. They are not close, they are **identical**, and that is not a coincidence. Q4 finds out why.

</details>

---

### Q4 · Why the two numbers are the same

Print how many test days the model predicted as a jump, and the smallest and largest probability it gave. Then say, in a comment, why the accuracy had to equal the majority rule.

In [ ]:
called = ...

print('days called a jump:', ...)
print('lowest probability :', ...)
print('highest probability:', ...)

# why is the accuracy exactly the majority rule?

<details>
<summary>💡 Hint</summary>

`called = jump_model.predict(test[['vol_20d']])`, then `called.sum()`. Compare the largest probability with one half.

</details>

<details>
<summary>✅ Solution</summary>

```python
called = jump_model.predict(test[['vol_20d']])

print('days called a jump:', int(called.sum()))
print('lowest probability :', round(float(p_jump.min()), 4))
print('highest probability:', round(float(p_jump.max()), 4))

# No test day reaches a probability of 0.5, so .predict() returns 0 on every
# day. Predicting 0 everywhere IS the majority rule, so the two accuracies
# are the same number by construction.
```

0 days called, with probabilities running from 0.0494 to 0.4876. The highest is below one half, so `.predict()` says no on every single day. The model has not failed to learn; it has been asked a question at a threshold that makes its answer always the same.

</details>

---

### Q5 · The ranking underneath

Score the same probabilities with the AUC and with the average precision, and print the share of jumps beside the second one. Store the AUC as `auc_jump`.

In [ ]:
auc_jump = ...
ap_jump = ...

print('AUC              :', auc_jump)
print('average precision:', ap_jump)
print('share of jumps   :', ...)

<details>
<summary>💡 Hint</summary>

`roc_auc_score(test['jump'], p_jump)` and `average_precision_score(test['jump'], p_jump)`. The no-information line for average precision is the share of ones, not 0.5.

</details>

<details>
<summary>✅ Solution</summary>

```python
auc_jump = round(roc_auc_score(test['jump'], p_jump), 4)
ap_jump = round(average_precision_score(test['jump'], p_jump), 4)

print('AUC              :', auc_jump)
print('average precision:', ap_jump)
print('share of jumps   :', round(test['jump'].mean(), 4))
```

An AUC of 0.8892 and an average precision of 0.4336 against 0.0871 for guessing. The ranking is better than Part 8's 0.797 on the easier label: the model knows perfectly well which days are dangerous. Everything that went wrong in Q3 and Q4 was the threshold, and the threshold is not part of the model.

</details>

---

### Q6 · Putting a price on the two mistakes

The desk prices a missed jump at 5 and a false alarm at 1. Sweep the threshold from 0.05 to 0.60 in steps of 0.01 **on the training rows**, pick the cheapest, and store it as `chosen`. Print it beside the value the cost formula gives.

In [ ]:
p_train = ...
grid = np.arange(0.05, 0.60, 0.01)
costs = []

for threshold in grid:
    ...

chosen = ...

print('chosen on the training rows:', chosen)
print('the formula says       :', round(1 / (1 + 5), 3))

<details>
<summary>💡 Hint 1</summary>

`p_train = jump_model.predict_proba(train[['vol_20d']])[:, 1]`. Inside the loop, build the 0/1 calls, unpack `confusion_matrix(...).ravel()` into `tn, fp, fn, tp`, and append `5 * fn + fp`.

</details>

<details>
<summary>💡 Hint 2</summary>

`chosen = round(float(grid[np.array(costs).argmin()]), 2)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
p_train = jump_model.predict_proba(train[['vol_20d']])[:, 1]
grid = np.arange(0.05, 0.60, 0.01)
costs = []

for threshold in grid:
    tn, fp, fn, tp = confusion_matrix(train['jump'], (p_train >= threshold).astype(int)).ravel()
    costs.append(5 * fn + fp)

chosen = round(float(grid[np.array(costs).argmin()]), 2)

print('chosen on the training rows:', chosen)
print('the formula says       :', round(1 / (1 + 5), 3))
```

The training rows choose 0.25, against 0.167 from the formula. The two differ because the formula assumes the probabilities are calibrated, and Q9 shows that on this label they are not. Either way, both are far below one half.

</details>

---

### Q7 · What the chosen threshold does

Apply `chosen` to the test probabilities. Print the confusion matrix, the recall, the precision, the accuracy, and the cost, and compare the cost with the cost at one half.

In [ ]:
called_chosen = ...
called_half = ...
matrix = ...
cost_chosen = ...
cost_half = ...

print(matrix)
print('recall   :', ...)
print('precision:', ...)
print('accuracy :', ...)
print('cost at the chosen threshold:', cost_chosen)
print('cost at one half            :', cost_half)

<details>
<summary>💡 Hint 1</summary>

`called_chosen = (p_jump >= chosen).astype(int)` and `called_half = (p_jump >= 0.5).astype(int)`.

</details>

<details>
<summary>💡 Hint 2</summary>

For each cost, unpack `confusion_matrix(test['jump'], called).ravel()` into `tn, fp, fn, tp` and compute `5 * fn + fp`.

</details>

<details>
<summary>✅ Solution</summary>

```python
called_chosen = (p_jump >= chosen).astype(int)
called_half = (p_jump >= 0.5).astype(int)
matrix = confusion_matrix(test['jump'], called_chosen)

tn, fp, fn, tp = matrix.ravel()
tn2, fp2, fn2, tp2 = confusion_matrix(test['jump'], called_half).ravel()
cost_chosen = 5 * fn + fp
cost_half = 5 * fn2 + fp2

print(matrix)
print('recall   :', round(recall_score(test['jump'], called_chosen), 4))
print('precision:', round(precision_score(test['jump'], called_chosen, zero_division=0), 4))
print('accuracy :', round(accuracy_score(test['jump'], called_chosen), 4))
print('cost at the chosen threshold:', cost_chosen)
print('cost at one half            :', cost_half)
```

At 0.25 the model calls 196 of the 482 test days and catches 38 of the 42 jumps, so the recall is 0.905 and the precision 0.194. The accuracy has **fallen** from 0.913 to 0.664, well below the majority rule, while the cost has fallen from 210 to 178. The model got worse by the number Part 8 reported and better by the number the desk actually pays.

</details>

---

### Q8 · The other way to move the cut

Fit the same pipeline again with `class_weight='balanced'` on the classifier, and print its accuracy, recall, how many days it calls, and its AUC. Compare each with the plain model's.

In [ ]:
weighted = ...
...
pred_w = ...

print('accuracy:', ..., 'was', acc_jump)
print('recall  :', ...)
print('days called:', ...)
print('AUC     :', ..., 'was', auc_jump)

<details>
<summary>💡 Hint</summary>

`LogisticRegression(class_weight='balanced')` inside the pipeline. The AUC still needs `predict_proba(...)[:, 1]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
weighted = Pipeline([('scale', StandardScaler()),
                     ('logit', LogisticRegression(class_weight='balanced'))])
weighted.fit(train[['vol_20d']], train['jump'])
pred_w = weighted.predict(test[['vol_20d']])

print('accuracy:', round(accuracy_score(test['jump'], pred_w), 4), 'was', acc_jump)
print('recall  :', round(recall_score(test['jump'], pred_w), 4))
print('days called:', int(pred_w.sum()))
print('AUC     :', round(roc_auc_score(test['jump'], weighted.predict_proba(test[['vol_20d']])[:, 1]), 4),
      'was', auc_jump)
```

The weights catch every one of the 42 jumps, a recall of 1.0, by calling 294 of the 482 days, and the accuracy falls to 0.4772. The AUC is 0.8892, the same as before: the weights moved the cut, not the ranking. A threshold does the same job and can be set to any value, rather than the one the class sizes happen to imply.

</details>

---

### Q9 · Do the probabilities mean what they say

Put the test days into the buckets 0 to 0.1, 0.1 to 0.2, 0.2 to 0.3 and 0.3 to 0.5 with `pd.cut`, and report how many days are in each and what share of them jumped. Then print the average probability beside the share that jumped.

In [ ]:
checked = ...
by_bucket = ...

print(by_bucket)

print('average probability:', ...)
print('share that jumped  :', ...)

<details>
<summary>💡 Hint 1</summary>

`pd.cut(checked['p'], [0, 0.1, 0.2, 0.3, 0.5])`.

</details>

<details>
<summary>💡 Hint 2</summary>

`checked.groupby('bucket', observed=True)['jump'].agg(['size', 'mean']).round(3)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
checked = test.copy()
checked['p'] = p_jump
checked['bucket'] = pd.cut(checked['p'], [0, 0.1, 0.2, 0.3, 0.5])
by_bucket = checked.groupby('bucket', observed=True)['jump'].agg(['size', 'mean']).round(3)

print(by_bucket)

print('average probability:', round(float(p_jump.mean()), 4))
print('share that jumped  :', round(float(test['jump'].mean()), 4))
```

The model gives 0.2337 on average where 0.0871 of the days jumped, so it overstates by a factor of nearly three. Of the 145 days it put between 0.3 and 0.5, 0.255 jumped; of the 181 days between 0.1 and 0.2, none did. The ranking is right and the level is wrong.

</details>

---

### Q10 · Whether calibration can repair it

Print the Brier score of the plain model and of the weighted one from Q8. Then wrap the plain pipeline in `CalibratedClassifierCV` with `method='sigmoid'` and `cv=5`, fit it on the training rows, and print its average probability and Brier score.

In [ ]:
print('plain   :', ...)
print('weighted:', ...)

fixed = ...
...
p_fixed = ...

print('calibrated average:', ...)
print('calibrated Brier  :', ...)

<details>
<summary>💡 Hint 1</summary>

`brier_score_loss(test['jump'], p_jump)`, and the same with the weighted model's probabilities.

</details>

<details>
<summary>💡 Hint 2</summary>

`CalibratedClassifierCV(Pipeline([...]), method='sigmoid', cv=5)`, fitted on `train[['vol_20d']]` and `train['jump']`.

</details>

<details>
<summary>✅ Solution</summary>

```python
print('plain   :', round(brier_score_loss(test['jump'], p_jump), 4))
print('weighted:', round(brier_score_loss(test['jump'],
      weighted.predict_proba(test[['vol_20d']])[:, 1]), 4))

fixed = CalibratedClassifierCV(
    Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())]),
    method='sigmoid', cv=5)
fixed.fit(train[['vol_20d']], train['jump'])
p_fixed = fixed.predict_proba(test[['vol_20d']])[:, 1]

print('calibrated average:', round(float(p_fixed.mean()), 4))
print('calibrated Brier  :', round(brier_score_loss(test['jump'], p_fixed), 4))
```

0.0876 for the plain model and 0.2757 for the weighted one, and calibrating moves the average only from 0.2337 to 0.2321 and the Brier score from 0.0876 to 0.0852. This is worth sitting with. `CalibratedClassifierCV` learns its correction on the training years, where 18.5% of days jumped, and applies it to test years where 8.7% did. The model is not miscalibrated because it was built wrongly; it is miscalibrated because the world got calmer, and no amount of arithmetic on the training rows can know that.

</details>

---

### Q11 · Twenty columns, one more time

Search `C` over `[0.0001, 0.001, 0.01, 0.1, 1, 10]` on the twenty columns with the time folds and `scoring='roc_auc'`, and print the winning `C`, its mean fold score, and its test AUC beside the one-column model's.

In [ ]:
grid = ...
search = ...
...

auc_wide = ...

print('best C  :', ...)
print('fold AUC:', ...)
print('test AUC:', auc_wide, 'against', auc_jump, 'for one column')

<details>
<summary>💡 Hint 1</summary>

The grid key is the step name, two underscores, the argument: `{'logit__C': [...]}`. Use `max_iter=1000` on the classifier.

</details>

<details>
<summary>💡 Hint 2</summary>

`search.best_params_`, `search.best_score_`, and `roc_auc_score(test['jump'], search.predict_proba(test[columns])[:, 1])`.

</details>

<details>
<summary>✅ Solution</summary>

```python
grid = {'logit__C': [0.0001, 0.001, 0.01, 0.1, 1, 10]}
search = GridSearchCV(
    Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression(max_iter=1000))]),
    grid, cv=folds, scoring='roc_auc')
search.fit(train[columns], train['jump'])

auc_wide = round(roc_auc_score(test['jump'], search.predict_proba(test[columns])[:, 1]), 4)

print('best C  :', search.best_params_)
print('fold AUC:', round(search.best_score_, 4))
print('test AUC:', auc_wide, 'against', auc_jump, 'for one column')
```

C of 0.01 wins the folds with 0.7547, and scores 0.7458 on the test days against 0.8892 for the single column. The twenty columns lose on the jump label exactly as they lost on the volatility forecast in Part 6 and on the rising label in Part 8. Three parts, three answers, the same answer.

</details>

---

### Q12 · Three answers instead of two

Build `move` with `pd.cut` on the ratio `vol_next / vol_20d`, with cuts at 0.85 and 1.25 and the names `falls`, `stays`, `rises`. Split again, fit the one-column pipeline with `max_iter=1000`, and print the accuracy, the baseline from the most common training label, and the macro F1.

In [ ]:
# add the ratio and the move label to table, then split again
...
train = ...
test = ...

three = ...
...
pred_move = ...

print('accuracy:', ...)
print('baseline:', ...)
print('macro F1:', ...)

<details>
<summary>💡 Hint 1</summary>

`pd.cut(ratio, [-np.inf, 0.85, 1.25, np.inf], labels=['falls', 'stays', 'rises'])`.

</details>

<details>
<summary>💡 Hint 2</summary>

The baseline is `(test['move'] == train['move'].value_counts().idxmax()).mean()`, and the macro F1 is `f1_score(test['move'], pred_move, average='macro')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
ratio = table['vol_next'] / table['vol_20d']
table['move'] = pd.cut(ratio, [-np.inf, 0.85, 1.25, np.inf],
                       labels=['falls', 'stays', 'rises'])
train = table.loc[:'2022-12-31']
test = table.loc['2023-01-01':]

three = Pipeline([('scale', StandardScaler()),
                  ('logit', LogisticRegression(max_iter=1000))])
three.fit(train[['vol_20d']], train['move'])
pred_move = three.predict(test[['vol_20d']])

print('accuracy:', round(accuracy_score(test['move'], pred_move), 4))
print('baseline:', round((test['move'] == train['move'].value_counts().idxmax()).mean(), 4))
print('macro F1:', round(f1_score(test['move'], pred_move, average='macro'), 4))
```

0.4315 against a baseline of 0.3755, with a macro F1 of 0.4169. Recall is 0.961 for `rises` but only 0.210 for `falls`: on Apple the model finds the busy months and cannot pick out the calm ones, which is the opposite of what the index did in the lecture.

</details>

---

### Q13 · A second classifier on the same folds

Search `k` over `[1, 5, 15, 51, 101, 151, 201, 301]` for `KNeighborsClassifier` in a scaled pipeline, on `vol_20d` predicting `jump`, with the same folds and `scoring='roc_auc'`. Print the winning `k`, its fold score and its test AUC, and put the logistic model's fold score beside it.

In [ ]:
# put the jump label back on table and split again
...
train = ...
test = ...

knn_search = ...
...
logistic_folds = ...

print('best k   :', ...)
print('k-NN folds:', ...)
print('k-NN test :', ...)
print('logistic folds:', ...)

<details>
<summary>💡 Hint 1</summary>

`GridSearchCV(Pipeline([('scale', StandardScaler()), ('knn', KNeighborsClassifier())]), {'knn__n_neighbors': [...]}, cv=folds, scoring='roc_auc')`.

</details>

<details>
<summary>💡 Hint 2</summary>

`logistic_folds = cross_val_score(Pipeline([...LogisticRegression()...]), train[['vol_20d']], train['jump'], cv=folds, scoring='roc_auc').mean()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
table['jump'] = (table['vol_next'] > 1.5 * table['vol_20d']).astype(int)
train = table.loc[:'2022-12-31']
test = table.loc['2023-01-01':]

knn_search = GridSearchCV(
    Pipeline([('scale', StandardScaler()), ('knn', KNeighborsClassifier())]),
    {'knn__n_neighbors': [1, 5, 15, 51, 101, 151, 201, 301]},
    cv=folds, scoring='roc_auc')
knn_search.fit(train[['vol_20d']], train['jump'])

logistic_folds = cross_val_score(
    Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())]),
    train[['vol_20d']], train['jump'], cv=folds, scoring='roc_auc').mean()

print('best k   :', knn_search.best_params_)
print('k-NN folds:', round(knn_search.best_score_, 4))
print('k-NN test :', round(roc_auc_score(test['jump'],
      knn_search.best_estimator_.predict_proba(test[['vol_20d']])[:, 1]), 4))
print('logistic folds:', round(logistic_folds, 4))
```

k of 101 wins its own search with 0.7586 on the folds, against 0.7923 for logistic regression, so the folds keep logistic regression. On the test days it is 0.8378 against 0.8892, the same ordering. One column and a boundary that is a single number is all this problem has ever needed.

</details>

---

### Q14 · Across the desk

For every instrument, build the same one-column jump model and collect three things in dictionaries: the share of test days that jumped, the test AUC, and whether the model calls no jump at all at a threshold of one half. Print the count of silent instruments and the AUCs sorted, largest first.

In [ ]:
desk_share = {}
desk_auc = {}
silent = []

for t in TICKERS:
    ...

print('silent at 0.5:', len(silent), 'of', len(TICKERS))
print(silent)
for t in sorted(desk_auc, key=desk_auc.get, reverse=True):
    print(t, desk_auc[t])

<details>
<summary>💡 Hint 1</summary>

Inside the loop, rebuild the two columns for that ticker: `vol_20d` is `rets[t].rolling(20).std()` and `vol_next` is the same shifted by -20, then `dropna()` and the 1.5 comparison.

</details>

<details>
<summary>💡 Hint 2</summary>

`if model_t.predict(te[['vol_20d']]).sum() == 0: silent.append(t)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
desk_share = {}
desk_auc = {}
silent = []

for t in TICKERS:
    frame = pd.DataFrame({'vol_20d': rets[t].rolling(20).std()})
    frame['vol_next'] = rets[t].rolling(20).std().shift(-20)
    frame = frame.dropna()
    frame['jump'] = (frame['vol_next'] > 1.5 * frame['vol_20d']).astype(int)
    tr = frame.loc[:'2022-12-31']
    te = frame.loc['2023-01-01':]
    model_t = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
    model_t.fit(tr[['vol_20d']], tr['jump'])
    desk_share[t] = round(float(te['jump'].mean()), 3)
    desk_auc[t] = round(float(roc_auc_score(te['jump'],
                        model_t.predict_proba(te[['vol_20d']])[:, 1])), 3)
    if model_t.predict(te[['vol_20d']]).sum() == 0:
        silent.append(t)

print('silent at 0.5:', len(silent), 'of', len(TICKERS))
print(silent)
for t in sorted(desk_auc, key=desk_auc.get, reverse=True):
    print(t, desk_auc[t])
```

9 of the 11 instruments call no jump at all at a threshold of one half, and 7 of the 11 rank at 0.75 or better, from 0.972 on MSFT down to 0.610 on JPM. Apple was not a special case: the default threshold silences most of the desk, and the ranking is usable on most of it.

</details>

---

### Q15 · Draw the decision

Draw the test probabilities over time as a line, with a horizontal line at `chosen` and another at 0.5, and mark the days that actually jumped. Label both axes and give the figure a title.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.4))

...
...
...

ax.set_xlabel('date')
ax.set_ylabel('probability of a jump')
plt.show()

<details>
<summary>💡 Hint 1</summary>

`ax.plot(test.index, p_jump)` for the line, and two `ax.axhline(...)` calls for the thresholds.

</details>

<details>
<summary>💡 Hint 2</summary>

For the jumps: `days = test.index[test['jump'] == 1]`, then `ax.scatter(days, p_jump[test['jump'].values == 1], s=14, color='#b3402f')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
fig, ax = plt.subplots(figsize=(10, 3.4))

ax.plot(test.index, p_jump, linewidth=1.4, color='#1c5cab')
days = test.index[test['jump'] == 1]
ax.scatter(days, p_jump[test['jump'].values == 1], s=14, color='#b3402f', zorder=3)
ax.axhline(chosen, color='#b8860b', linestyle='--')
ax.axhline(0.5, color='grey', linestyle=':')

ax.set_xlabel('date')
ax.set_ylabel('probability of a jump')
ax.set_title('Apple: the probability of a jump, the chosen threshold, and the jumps',
             loc='left')
plt.show()
```

The dotted line at one half sits above everything, which is Q4 as a picture. The dashed line at 0.25 cuts through the series, and most of the marked jumps sit above it. The probabilities move slowly, so the warnings come in runs rather than one day at a time.

</details>

---

### Q16 · Write down what you would defend

Collect what Part 9 established into a dictionary `report`, and write `summarise(report)` printing one line per entry and ending with a verdict: whether the desk should run this model on Apple, and at what threshold.

In [ ]:
report = {
    'label': ...,
    'auc': ...,
    'accuracy_at_half': ...,
    'days_called_at_half': ...,
    'chosen_threshold': ...,
    'recall_at_chosen': ...,
    'cost_at_chosen': ...,
    'cost_at_half': ...,
    'calibrated': ...,
    'silent_instruments': ...,
}


def summarise(r):
    """..."""
    ...


summarise(report)

<details>
<summary>💡 Hint 1</summary>

Every value is a number or a short string you already printed. For `calibrated`, a short string such as `'no: says 0.23 where 0.09 happens'`.

</details>

<details>
<summary>💡 Hint 2</summary>

Inside the function, `for key, value in r.items():` and a `print` with an f-string, then an `if` on the AUC for the verdict.

</details>

<details>
<summary>✅ Solution</summary>

```python
report = {
    'label': 'jump: vol_next > 1.5 x vol_20d',
    'auc': auc_jump,
    'accuracy_at_half': acc_jump,
    'days_called_at_half': 0,
    'chosen_threshold': chosen,
    'recall_at_chosen': 0.905,
    'cost_at_chosen': 178,
    'cost_at_half': 210,
    'calibrated': 'no: says 0.23 on average where 0.09 happens',
    'silent_instruments': '9 of 11 at a threshold of 0.5',
}


def summarise(r):
    """Print the report and the verdict it supports."""
    for key, value in r.items():
        print(f"{key:<22}{value}")
    print()
    if r['auc'] >= 0.75:
        print(f"Verdict: the ranking is usable. Run it at {r['chosen_threshold']},")
        print("not at one half, and treat the probabilities as an order, not a level.")
    else:
        print('Verdict: the ranking is too weak to act on.')


summarise(report)
```

The honest report has two halves that point in opposite directions. The model ranks Apple's dangerous months well, with an AUC of 0.889 and 7 of 11 instruments above 0.75. It is also unusable as shipped: at one half it predicts nothing, its probabilities run about three times too high, and calibrating on the training years cannot fix that. What you defend is the ranking plus a threshold of 0.25 chosen from the desk's own costs, reviewed whenever the costs change.

</details>

---

## 📦 What you now have

| what | where |
|:--|:--|
| the table and split carried from Part 8 | `table`, `columns`, `train`, `test` |
| the rare label and its baseline | `table['jump']`, `majority` |
| the one-column classifier and its probabilities | `jump_model`, `p_jump`, `auc_jump`, `acc_jump` |
| the threshold chosen from costs | `chosen`, `called_chosen` |
| the weighted alternative | `weighted`, `pred_w` |
| the calibration check | `by_bucket`, `fixed`, `p_fixed` |
| twenty columns with C chosen | `search`, `auc_wide` |
| three classes | `table['move']`, `three`, `pred_move` |
| the second classifier | `knn_search`, `logistic_folds` |
| the whole desk | `desk_share`, `desk_auc`, `silent` |
| the thing you would defend | `report` |

## What changed since Part 8

- **The label became rare.** 8.7% of test days rather than about half, so the rule to beat rose from 0.546 to 0.9129.
- **Accuracy stopped working.** The model scores 0.9129, exactly the majority rule, while predicting no jump on any day, because no probability reaches one half (Q3 and Q4). Its AUC is 0.889.
- **The threshold became a decision.** Priced at 5 for a miss and 1 for a false alarm, the training rows choose 0.25, which catches 90% of the jumps and cuts the cost from 210 to 178 (Q6 and Q7).
- **The probabilities are not calibrated, and cannot be fixed here.** The model says 0.23 on average where 0.09 happens, because 18% of the training days jumped against 9% of the test days (Q9 and Q10).
- **The wide model lost again.** 0.746 against 0.889 with C chosen on the folds (Q11), as in Parts 6 and 8.
- **The desk agrees.** 9 of 11 instruments are silent at one half, and 7 of 11 rank at 0.75 or better (Q14).

## Where this leaves the risk report

The report can now say three separate things and keep them separate: how well the model ranks months by danger, where the desk has chosen to act given what each mistake costs it, and how much trust the probability itself deserves. Part 8 could only say the first. The third is the uncomfortable one on this data, because the answer is that the level cannot be trusted on a period when the base rate has moved.

**Next part:** the whole classification workflow on one instrument from beginning to end, as a single investigation rather than a sequence of questions.